In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- Configuration ---
INPUT_PREPROCESSED_DATA_PATH = '/content/drive/MyDrive/NLP_Project_Preprocessed_Features.csv'
OUTPUT_BASE_DIR = '/content/drive/MyDrive/NLP_Project_Final_Data'

# Define the split ratios (e.g., 80% Train, 10% Validation, 10% Test)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
# Ensure ratios sum to 1.0 (or very close)
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Ratios must sum to 1.0"

# Target column name (CAMEO code is our classification target)
TARGET_COLUMN = 'CAMEO_CODE'

# Columns to keep for the final modeling input
FINAL_COLUMNS = [
    'SentenceText',
    'CAMEO_CODE',
    'NER_Features',
    'Temporal_Features',
    'ArticleURL'
]


def run_final_preparation():
    """Loads preprocessed data, performs final filtering, EDA, and splits the data."""
    print(f"Loading preprocessed features from: {INPUT_PREPROCESSED_DATA_PATH}")
    try:
        df = pd.read_csv(INPUT_PREPROCESSED_DATA_PATH)
    except FileNotFoundError:
        print(f"ERROR: Input file not found at {INPUT_PREPROCESSED_DATA_PATH}. Check your path.")
        return

    # --- 1. Initial Filtering and Cleanup ---
    # Ensure the target column is present and not null
    initial_count = len(df)
    df.dropna(subset=[TARGET_COLUMN, 'SentenceText'], inplace=True)
    df = df[df['SentenceText'].str.strip().str.len() > 20] # Minimum sentence length check

    # Convert CAMEO code to string to ensure consistent handling
    df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str)

    print(f"Removed {initial_count - len(df)} rows with missing data or short sentences.")
    print(f"Total clean sentence records for splitting: {len(df)}")

    # --- 2. Exploratory Data Analysis (EDA) on Labels ---
    print("\n--- EDA: CAMEO Code Distribution (Top 10) ---")
    label_counts = df[TARGET_COLUMN].value_counts()
    print(label_counts.head(10))
    print(f"Total unique CAMEO codes (classes): {len(label_counts)}")

    # Calculate sentence length statistics
    df['SentenceLength'] = df['SentenceText'].apply(len)
    print("\n--- EDA: Sentence Length Statistics (Characters) ---")
    print(df['SentenceLength'].describe())

    # --- 3. Data Splitting (Stratified by CAMEO Code) ---
    print("\n--- Splitting Data into Train, Validation, and Test Sets ---")

    # Because we have many classes, we only stratify on CAMEO codes that appear frequently.
    # Rare codes are grouped into a 'Non-Stratified' category to prevent errors.

    # Find codes that appear enough times to be in all splits (e.g., at least 5 times)
    min_samples = int(1 / min(TRAIN_RATIO, VAL_RATIO, TEST_RATIO) * 1.5) # Minimum 15-20 samples

    frequent_labels = label_counts[label_counts >= min_samples].index.tolist()

    df['StratifyLabel'] = np.where(
        df[TARGET_COLUMN].isin(frequent_labels),
        df[TARGET_COLUMN],
        'RARE_CLASS'
    )

    stratify_col = df['StratifyLabel']

    # First split: 80% Train, 20% Temp (Val + Test)
    df_train, df_temp = train_test_split(
        df,
        test_size=(VAL_RATIO + TEST_RATIO),
        random_state=42,
        stratify=stratify_col
    )

    # Calculate new ratios for the second split (e.g., 10% Val / 20% Temp = 0.5)
    test_val_ratio_in_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)

    # Second split: 50% Val, 50% Test from Temp set
    df_val, df_test = train_test_split(
        df_temp,
        test_size=test_val_ratio_in_temp, # This is the fraction for validation
        random_state=42,
        # Stratify using the temporary set's labels
        stratify=df_temp['StratifyLabel']
    )

    # Final check of sizes
    print(f"Total Train Records: {len(df_train)} ({len(df_train)/len(df):.2f})")
    print(f"Total Validation Records: {len(df_val)} ({len(df_val)/len(df):.2f})")
    print(f"Total Test Records: {len(df_test)} ({len(df_test)/len(df):.2f})")

    # --- 4. Final Save ---
    os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

    # Select only the required columns before saving
    df_train[FINAL_COLUMNS].to_csv(os.path.join(OUTPUT_BASE_DIR, 'train_data.csv'), index=False)
    df_val[FINAL_COLUMNS].to_csv(os.path.join(OUTPUT_BASE_DIR, 'val_data.csv'), index=False)
    df_test[FINAL_COLUMNS].to_csv(os.path.join(OUTPUT_BASE_DIR, 'test_data.csv'), index=False)

    print("\n--- SUCCESS: Final Datasets Saved ---")
    print(f"Train data saved to: {os.path.join(OUTPUT_BASE_DIR, 'train_data.csv')}")
    print(f"Validation data saved to: {os.path.join(OUTPUT_BASE_DIR, 'val_data.csv')}")
    print(f"Test data saved to: {os.path.join(OUTPUT_BASE_DIR, 'test_data.csv')}")
    print("\nYour data is now ready for model fine-tuning!")


if __name__ == '__main__':
    run_final_preparation()

Loading preprocessed features from: /content/drive/MyDrive/NLP_Project_Preprocessed_Features.csv
Removed 4645 rows with missing data or short sentences.
Total clean sentence records for splitting: 2800994

--- EDA: CAMEO Code Distribution (Top 10) ---
CAMEO_CODE
190     613097
173     560538
130     344337
172     229039
193     195627
192     128967
141     128146
180     121727
138      50195
1712     41794
Name: count, dtype: int64
Total unique CAMEO codes (classes): 59

--- EDA: Sentence Length Statistics (Characters) ---
count    2.800994e+06
mean     1.688031e+02
std      2.895425e+02
min      2.100000e+01
25%      7.900000e+01
50%      1.250000e+02
75%      1.850000e+02
max      1.250170e+05
Name: SentenceLength, dtype: float64

--- Splitting Data into Train, Validation, and Test Sets ---
Total Train Records: 2240795 (0.80)
Total Validation Records: 280099 (0.10)
Total Test Records: 280100 (0.10)

--- SUCCESS: Final Datasets Saved ---
Train data saved to: /content/drive/MyDrive/

In [4]:
# Install the Hugging Face Transformers library and PyTorch utilities
!pip install transformers accelerate datasets
# Install deep learning utilities like Pytorch Lighting (optional but useful)
!pip install pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 36.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import json
import os
from sklearn.preprocessing import LabelEncoder
import numpy as np # Import numpy to explicitly handle numerical types

# --- Configuration ---
TRAIN_DATA_PATH = '/content/drive/MyDrive/NLP_Project_Final_Data/train_data.csv'
CONFIG_OUTPUT_PATH = '/content/drive/MyDrive/NLP_Project_Final_Data/label_map.json'
CAMEO_CODE_COLUMN = 'CAMEO_CODE'

# Custom Encoder to handle NumPy types during JSON serialization
class NpEncoder(json.JSONEncoder):
    """Ensures all NumPy types are converted to native Python types for JSON serialization."""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

def create_and_save_label_map():
    """
    Loads the training data, creates a mapping from CAMEO codes (strings)
    to integer IDs (0 to N-1), and saves it as a JSON file.
    """
    print(f"Loading training data from: {TRAIN_DATA_PATH}")
    try:
        # CRITICAL FIX 1: Ensure CAMEO_CODE column is read as a string type from the start
        df_train = pd.read_csv(TRAIN_DATA_PATH, usecols=[CAMEO_CODE_COLUMN], dtype={CAMEO_CODE_COLUMN: str})
    except FileNotFoundError:
        print(f"ERROR: Training data not found at {TRAIN_DATA_PATH}. Aborting map creation.")
        return

    # --- 1. Initialize and Fit the LabelEncoder ---
    le = LabelEncoder()
    # Fit the encoder to all unique CAMEO codes in the training set
    le.fit(df_train[CAMEO_CODE_COLUMN].unique())

    # CRITICAL FIX 2: Ensure all labels are explicitly native Python strings
    native_python_labels = [str(label) for label in le.classes_]

    # --- 2. Create the Final Mappings ---

    # map_id_to_label: {0: '190', 1: '173', ...}
    # Keys are Python int (temporarily), values are Python str
    id_to_label = {int(i): label for i, label in enumerate(native_python_labels)}

    # map_label_to_id: {'190': 0, '173': 1, ...}
    # Keys are Python str, values are Python int
    label_to_id = {label: int(i) for i, label in enumerate(native_python_labels)}

    # Ensure num_labels is a standard Python int
    num_labels = int(len(native_python_labels))

    print(f"Successfully mapped {num_labels} unique CAMEO codes.")

    # --- 3. Save Configuration to JSON ---
    config = {
        "num_labels": num_labels,
        "id2label": id_to_label,
        "label2id": label_to_id,
        "model_name": "roberta-base",
        "max_seq_length": 256
    }

    # Convert integer keys in id2label to strings for the final JSON structure
    # (Hugging Face expects string keys for id2label in config)
    config['id2label'] = {str(k): v for k, v in config['id2label'].items()}

    with open(CONFIG_OUTPUT_PATH, 'w') as f:
        # Use the custom encoder to ensure all NumPy types are safely converted
        # This will now succeed as all keys are guaranteed to be Python strings.
        json.dump(config, f, indent=4, cls=NpEncoder)

    print(f"Configuration file saved to: {CONFIG_OUTPUT_PATH}")
    print("\nReady for Model Fine-Tuning Script.")

if __name__ == '__main__':
    create_and_save_label_map()

Loading training data from: /content/drive/MyDrive/NLP_Project_Final_Data/train_data.csv
Successfully mapped 59 unique CAMEO codes.
Configuration file saved to: /content/drive/MyDrive/NLP_Project_Final_Data/label_map.json

Ready for Model Fine-Tuning Script.


In [ ]:
# import pandas as pd
# import numpy as np
# import json
# import os
# from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
# from sklearn.preprocessing import LabelEncoder

# # Hugging Face and PyTorch imports
# from datasets import Dataset
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
# import torch

# # --- Configuration (MUST MATCH PREVIOUS STEPS) ---
# BASE_DIR = '/content/drive/MyDrive/NLP_Project_Final_Data'
# LABEL_MAP_PATH = os.path.join(BASE_DIR, 'label_map.json')
# TRAIN_FILE = os.path.join(BASE_DIR, 'train_data.csv')
# VAL_FILE = os.path.join(BASE_DIR, 'val_data.csv')
# TEST_FILE = os.path.join(BASE_DIR, 'test_data.csv')
# OUTPUT_DIR = '/content/drive/MyDrive/NLP_Project_Model_Output'

# # Load Configuration Details (Model Name, Labels, Max Length)
# with open(LABEL_MAP_PATH, 'r') as f:
#     CONFIG = json.load(f)

# MODEL_NAME = CONFIG['model_name'] # "roberta-base"
# NUM_LABELS = CONFIG['num_labels'] # 59
# MAX_SEQ_LENGTH = CONFIG['max_seq_length'] # 256
# ID2LABEL = {int(k): v for k, v in CONFIG['id2label'].items()}
# LABEL2ID = CONFIG['label2id']


# def load_and_preprocess_data():
#     """Loads CSVs, converts CAMEO codes to numerical IDs, and creates Hugging Face Datasets."""
#     print("Loading datasets...")
#     df_train = pd.read_csv(TRAIN_FILE)
#     df_val = pd.read_csv(VAL_FILE)
#     df_test = pd.read_csv(TEST_FILE)

#     # 1. Convert string labels (CAMEO codes) to numerical IDs (0 to 58)
#     # This step is critical as PyTorch models require integer labels.
#     df_train['labels'] = df_train['CAMEO_CODE'].astype(str).map(LABEL2ID)
#     df_val['labels'] = df_val['CAMEO_CODE'].astype(str).map(LABEL2ID)
#     df_test['labels'] = df_test['CAMEO_CODE'].astype(str).map(LABEL2ID)

#     # Drop rows where mapping failed (should be none, but safe check)
#     df_train.dropna(subset=['labels'], inplace=True)

#     print(f"Train samples: {len(df_train)}, Val samples: {len(df_val)}, Test samples: {len(df_test)}")

#     # 2. Convert Pandas DataFrames to Hugging Face Dataset objects
#     # We only need 'SentenceText' as the input and 'labels' as the target.
#     train_dataset = Dataset.from_pandas(df_train[['SentenceText', 'labels']].reset_index(drop=True))
#     val_dataset = Dataset.from_pandas(df_val[['SentenceText', 'labels']].reset_index(drop=True))
#     test_dataset = Dataset.from_pandas(df_test[['SentenceText', 'labels']].reset_index(drop=True))

#     return train_dataset, val_dataset, test_dataset


# def tokenize_function(examples):
#     """Tokenizes text using the RoBERTa tokenizer."""
#     # Tokenizer is a global object, accessible here
#     return tokenizer(examples['SentenceText'], padding="max_length", truncation=True, max_length=MAX_SEQ_LENGTH)


# def compute_metrics(p):
#     """Custom function to calculate F1 score and Accuracy."""
#     # p is a named tuple: (predictions, label_ids)

#     # 1. Get predictions (highest probability class ID)
#     preds = np.argmax(p.predictions, axis=1)

#     # 2. Get true labels
#     labels = p.label_ids

#     # 3. Calculate metrics
#     accuracy = accuracy_score(labels, preds)
#     # Use 'weighted' average F1 because of the severe class imbalance (as seen in EDA)
#     f1_weighted = f1_score(labels, preds, average='weighted', zero_division=0)

#     # 4. Return as dictionary for Trainer logging
#     return {
#         'accuracy': accuracy,
#         'f1_weighted': f1_weighted,
#     }


# def main():
#     """The main fine-tuning loop."""
#     global tokenizer # Define tokenizer globally so tokenize_function can access it

#     # Ensure GPU is available
#     if not torch.cuda.is_available():
#         print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
#         print("!!! WARNING: GPU NOT DETECTED. TRAINING WILL BE EXTREMELY SLOW. !!!")
#         print("!!! Please go to Runtime -> Change runtime type and select GPU. !!!")
#         print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")

#     # 1. Load Tokenizer and Model
#     print(f"\nLoading tokenizer and model: {MODEL_NAME}...")
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#     # Load RoBERTa for Sequence Classification
#     model = AutoModelForSequenceClassification.from_pretrained(
#         MODEL_NAME,
#         num_labels=NUM_LABELS,
#         id2label=ID2LABEL,
#         label2id=LABEL2ID
#     )

#     # 2. Load and Prepare Data
#     train_dataset, val_dataset, test_dataset = load_and_preprocess_data()

#     # 3. Tokenize Data
#     print("Tokenizing datasets (This may take a minute for 2.8M records)...")
#     train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])
#     val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])

#     # Set the format for PyTorch
#     train_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])
#     val_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

#     # 4. Define Training Arguments
#     print("\nSetting up training arguments...")
#     training_args = TrainingArguments(
#         output_dir=OUTPUT_DIR,
#         num_train_epochs=3,                     # Number of training epochs
#         per_device_train_batch_size=32,         # Batch size per GPU/CPU
#         per_device_eval_batch_size=64,          # Evaluation batch size
#         warmup_steps=500,                       # Number of warmup steps for learning rate scheduler
#         weight_decay=0.01,                      # Strength of weight decay
#         logging_dir='./logs',
#         logging_steps=500,
#         eval_strategy="epoch",                  # **CORRECTED:** Use 'eval_strategy' instead of 'evaluation_strategy'
#         save_strategy="epoch",                  # Save checkpoint after each epoch
#         load_best_model_at_end=True,            # Load the best model found during training
#         metric_for_best_model="f1_weighted",    # Optimize for weighted F1 score
#         save_total_limit=2,                     # Only keep the last 2 best checkpoints
#         fp16=True,                              # Enable 16-bit precision training (if GPU is available)
#         gradient_accumulation_steps=2,          # Increase effective batch size to 64
#         learning_rate=2e-5,                     # Standard learning rate for fine-tuning
#     )

#     # 5. Initialize Trainer
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_tokenized,
#         eval_dataset=val_tokenized,
#         tokenizer=tokenizer,
#         compute_metrics=compute_metrics,
#     )

#     # 6. Start Fine-Tuning
#     print("\n" + "="*50)
#     print("Starting Fine-Tuning RoBERTa on CAMEO Classification Task")
#     print("="*50 + "\n")
#     trainer.train()

#     # 7. Save the final best model and tokenizer
#     print("\nTraining complete. Saving best model...")
#     final_model_path = os.path.join(OUTPUT_DIR, "best_model")
#     model.save_pretrained(final_model_path)
#     tokenizer.save_pretrained(final_model_path)
#     print(f"Final model saved to: {final_model_path}")

#     # 8. Final Evaluation on the unseen Test Set
#     print("\n" + "="*50)
#     print("Running Final Evaluation on Unseen Test Set")
#     print("="*50)

#     # Tokenize Test Set
#     test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])
#     test_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

#     # Run prediction
#     predictions = trainer.predict(test_tokenized)

#     # Extract metrics
#     test_metrics = compute_metrics(predictions)
#     print(f"\nTest Set Results:")
#     print(f"  -> Test Accuracy: {test_metrics['accuracy']:.4f}")
#     print(f"  -> Test F1 (Weighted): {test_metrics['f1_weighted']:.4f}")

#     # Optionally, calculate the confusion matrix for analysis
#     preds = np.argmax(predictions.predictions, axis=1)
#     labels = predictions.label_ids

#     # Note: Creating a confusion matrix for 59 classes can be huge.
#     # This is more for analysis after the fact, but we'll include the code structure.
#     # cm = confusion_matrix(labels, preds)
#     # print("\nConfusion Matrix (59x59 classes generated, but not printed due to size).")


# if __name__ == '__main__':
#     main()


Loading tokenizer and model: roberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...


Starting Fine-Tuning RoBERTa on CAMEO Classification Task



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: meghna-ganeshkumar (meghna-ganeshkumar-northwestern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,2.169900,2.142410,0.363079,0.325595
2,2.039400,2.057166,0.386299,0.357366


In [4]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Hugging Face and PyTorch imports
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

# --- Configuration (MUST MATCH PREVIOUS STEPS) ---
BASE_DIR = '/content/drive/MyDrive/NLP_Project_Final_Data'
LABEL_MAP_PATH = os.path.join(BASE_DIR, 'label_map.json')
TRAIN_FILE = os.path.join(BASE_DIR, 'train_data.csv')
VAL_FILE = os.path.join(BASE_DIR, 'val_data.csv')
TEST_FILE = os.path.join(BASE_DIR, 'test_data.csv')
OUTPUT_DIR = '/content/drive/MyDrive/NLP_Project_Model_Output'

# Define the exact checkpoint path to resume from
# This checkpoint contains the model state after Epoch 2 (checkpoint-70026)
RESUME_CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint-70026')


# Load Configuration Details (Model Name, Labels, Max Length)
with open(LABEL_MAP_PATH, 'r') as f:
    CONFIG = json.load(f)

MODEL_NAME = CONFIG['model_name'] # "roberta-base"
NUM_LABELS = CONFIG['num_labels'] # 59
MAX_SEQ_LENGTH = CONFIG['max_seq_length'] # 256
ID2LABEL = {int(k): v for k, v in CONFIG['id2label'].items()}
LABEL2ID = CONFIG['label2id']


def load_and_preprocess_data():
    """Loads CSVs, converts CAMEO codes to numerical IDs, and creates Hugging Face Datasets."""
    print("Loading datasets...")
    df_train = pd.read_csv(TRAIN_FILE)
    df_val = pd.read_csv(VAL_FILE)
    df_test = pd.read_csv(TEST_FILE)

    # 1. Convert string labels (CAMEO codes) to numerical IDs (0 to 58)
    df_train['labels'] = df_train['CAMEO_CODE'].astype(str).map(LABEL2ID)
    df_val['labels'] = df_val['CAMEO_CODE'].astype(str).map(LABEL2ID)
    df_test['labels'] = df_test['CAMEO_CODE'].astype(str).map(LABEL2ID)

    df_train.dropna(subset=['labels'], inplace=True)

    print(f"Train samples: {len(df_train)}, Val samples: {len(df_val)}, Test samples: {len(df_test)}")

    # 2. Convert Pandas DataFrames to Hugging Face Dataset objects
    train_dataset = Dataset.from_pandas(df_train[['SentenceText', 'labels']].reset_index(drop=True))
    val_dataset = Dataset.from_pandas(df_val[['SentenceText', 'labels']].reset_index(drop=True))
    test_dataset = Dataset.from_pandas(df_test[['SentenceText', 'labels']].reset_index(drop=True))

    return train_dataset, val_dataset, test_dataset


def tokenize_function(examples):
    """Tokenizes text using the RoBERTa tokenizer."""
    return tokenizer(examples['SentenceText'], padding="max_length", truncation=True, max_length=MAX_SEQ_LENGTH)


def compute_metrics(p):
    """Custom function to calculate F1 score and Accuracy."""
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    accuracy = accuracy_score(labels, preds)
    # Use 'weighted' average F1 because of the severe class imbalance
    f1_weighted = f1_score(labels, preds, average='weighted', zero_division=0)

    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
    }


def main():
    """The main fine-tuning loop."""
    global tokenizer

    if not torch.cuda.is_available():
        print("!!! WARNING: GPU NOT DETECTED. TRAINING WILL BE EXTREMELY SLOW. !!!")

    # 1. Load Tokenizer and Model
    # IMPORTANT: The model will be initialized from scratch here, but the
    # trainer.train(resume_from_checkpoint=...) call later will load the weights.
    print(f"\nLoading tokenizer and model: {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    # 2. Load and Prepare Data
    train_dataset, val_dataset, test_dataset = load_and_preprocess_data()

    # 3. Tokenize Data
    print("Tokenizing datasets (This may take a minute for 2.8M records)...")
    # This should hit the cache and be fast
    train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])
    val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])

    train_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])
    val_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

    # 4. Define Training Arguments
    print("\nSetting up training arguments...")
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,                     # Target 3 epochs total
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=500,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        save_total_limit=2,
        fp16=True,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        report_to="none",
    )

    # 5. Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # 6. Start Fine-Tuning, RESUMING from the last checkpoint
    print("\n" + "="*50)
    print(f"Starting Fine-Tuning RoBERTa, RESUMING from: {RESUME_CHECKPOINT_PATH}")
    print("Only the FINAL Epoch (Epoch 3) remains.")
    print("="*50 + "\n")

    # CRITICAL CHANGE: Resume training from the checkpoint folder
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_PATH)


    # 7. Save the final best model and tokenizer
    print("\nTraining complete. Saving best model...")
    final_model_path = os.path.join(OUTPUT_DIR, "best_model")
    model.save_pretrained(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    print(f"Final model saved to: {final_model_path}")

    # 8. Final Evaluation on the unseen Test Set
    print("\n" + "="*50)
    print("Running Final Evaluation on Unseen Test Set")
    print("="*50)

    # Tokenize Test Set
    test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=['SentenceText'])
    test_tokenized.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'])

    # Run prediction
    predictions = trainer.predict(test_tokenized)

    # Extract metrics
    test_metrics = compute_metrics(predictions)
    print(f"\nTest Set Results:")
    print(f"  -> Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"  -> Test F1 (Weighted): {test_metrics['f1_weighted']:.4f}")


if __name__ == '__main__':
    main()


Loading tokenizer and model: roberta-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

You are resuming training from a checkpoint trained with 4.57.1 of Transformers but your current version is 4.57.2. This is not recommended and could yield to errors or unwanted behaviors.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
3,1.943700,2.025921,0.396842,0.371280



Training complete. Saving best model...
Final model saved to: /content/drive/MyDrive/NLP_Project_Model_Output/best_model

Running Final Evaluation on Unseen Test Set


Map:   0%|          | 0/280100 [00:00<?, ? examples/s]


Test Set Results:
  -> Test Accuracy: 0.3968
  -> Test F1 (Weighted): 0.3706
